In [16]:
import random
import heapq


# type for a cell
type Cell =  tuple[int, int]

#size of the grid is 20x20
size = 20

#The maze
maze = [["."] * size for _ in range(size)]

#where the player starts
start: Cell = (0,0)

# exits
goal = (random.randint(0, size - 1), size - 1);

#the walls
walls = 0

# obstacle to passage ratio
ratio = 0.50 # half of the maze is obstacles


#the function to determine if the cell can have a wall
#it uses DFS to determine if the exits are reachable after
#placing a wall on the grid
def validWall() -> bool:
    stack: list[Cell] = [start]
    visited = {start}

    while stack:
        row, col = stack.pop();

        if (row, col) == goal:
            return True;

        # check the neighbors
        neighbor  = (row + 1, col)
        if (
                row + 1 < size
                and maze[row + 1][col] != "X"
                and neighbor not in visited
        ):
            visited.add(neighbor)
            stack.append(neighbor)

        neighbor  = (row - 1, col)
        if (
                row - 1 >= 0
                and maze[row - 1][col] != "X"
                and neighbor not in visited
        ):
            visited.add(neighbor)
            stack.append(neighbor)

        neighbor  = (row, col + 1)
        if (
                col + 1 < size
                and maze[row][col + 1] != "X"
                and neighbor not in visited
        ):
            visited.add(neighbor)
            stack.append(neighbor)

        neighbor  = (row, col - 1)
        if (
                col - 1 >= 0
                and maze[row][col - 1] != "X"
                and neighbor not in visited
        ):
            visited.add(neighbor)
            stack.append(neighbor)

    return False

while walls / (size * size) < ratio:

    #check if the maze is still reachable after placing the wall
    isValidWall = False;

    while not isValidWall:
        # keep generating a random cell in the grid until a valid wall cell
        cell = (
            random.randint(0, size - 1),
            random.randint(0, size - 1)
        )

        #skip the start and exit cells
        if cell == start or cell == goal:
            continue

        #skip cells that are already walls
        if maze[cell[0]][cell[1]] == "X":
            continue

        #place the wall in that cell temporarily
        maze[cell[0]][cell[1]] = "X"

        #update isValidWall
        isValidWall = validWall()

        #remove the wall if it is not a valid wall cell
        if not isValidWall:
            maze[cell[0]][cell[1]] = "."

    walls += 1


#place the goal on the grid
maze[goal[0]][goal[1]] = "#"

In [1]:
#The A* search

def aStar():
    # array that holds current cell, and its f(n) and g(n) values
    frontier = [(0, 0, start)]

    #the cells that were already explored
    visited = set()

    #holds the cell we came from for constructing the path later
    cameFrom = {}

    #map for holding the best g value for cell found so far
    gScore = {start: 0}

    while frontier:
        f, g, current = heapq.heappop(frontier)

        #if the cell had already been explored, skip it
        if current in visited:
             continue

         #add the cell to visited
        visited.add(current)

        #if goal is reached return the path
        if current == goal:
            return cameFrom, visited

        row, col = current

        neighbors = [
            (row + 1, col),
            (row - 1, col),
            (row, col + 1),
            (row, col - 1)
        ]

        for neighbor in neighbors:
            r,c = neighbor

            #for any cell we can move to calculate g(n) and f(n) for it
            #and add it to the heap
            if (
                    r < size and r >=0
                    and c < size and c >=0
                    and maze[r][c] != "X"
            ):
                newG = g + 1
                if neighbor not in gScore or gScore[neighbor] > newG:
                    gScore[neighbor] = newG

                    f = newG + abs(goal[0] - r) + abs(goal[1] - c)

                    # push it to the heap
                    heapq.heappush(frontier, (f, newG, neighbor))

                    #update cameform
                    cameFrom[neighbor] = current

    return None, visited

In [6]:
# Utility function to reconstruct the path return by A*
def reconstructPath(cameFrom):
    path = []
    current = goal;

    while current != start:
        path.append(current)
        current = cameFrom[current]

    path.append(start)
    path.reverse()

    return path

#function to print the maze as a formatted grid
def printMaze():
    for row in maze:
        print("  ".join(row))

In [20]:

#Print the Maze
printMaze()

#newline for formatting
print("\n") 


cameFrom, visited = aStar()
path = reconstructPath(cameFrom)

print(path, "\n")

print(f"The algorithm explored {len(visited)} states to find the goal at {goal}")

.  .  .  X  .  X  X  X  X  X  .  .  X  .  .  X  X  X  .  .
.  .  .  .  .  X  .  .  .  .  X  .  X  .  .  .  .  X  X  X
.  .  .  .  .  .  .  .  .  X  .  .  X  X  .  X  .  X  .  X
.  X  X  X  .  X  .  X  X  X  X  X  X  X  X  X  X  .  X  X
X  X  .  .  .  .  .  X  X  .  .  X  X  .  .  .  .  X  X  .
.  X  X  X  X  .  X  .  .  X  X  X  .  .  X  X  X  X  .  .
.  X  X  X  X  .  .  .  X  .  X  .  X  X  .  .  X  .  X  .
X  .  X  X  .  X  .  .  X  X  .  X  X  .  X  X  .  .  .  X
X  X  X  X  .  .  .  .  X  .  .  X  X  X  .  X  .  .  .  X
.  X  .  .  X  .  .  X  X  X  X  X  .  X  .  .  .  X  .  .
X  X  X  X  X  X  .  .  .  X  .  X  .  X  X  X  .  X  X  X
.  .  X  .  X  .  X  X  .  X  .  .  X  .  X  X  X  .  X  X
.  X  .  X  .  .  .  X  .  .  .  X  X  X  .  .  X  X  X  X
.  X  .  .  X  X  X  .  .  X  X  X  .  X  X  .  X  X  .  X
X  X  X  .  X  .  .  X  .  X  .  .  X  .  X  .  X  X  X  X
.  .  .  .  X  .  X  .  .  X  X  .  X  X  X  .  .  .  .  X
.  X  .  .  .  .  X  .  X  X  X  X  .  .  X  .  X  X  . 